In [1]:
 # Check GPU type
!nvidia-smi

Mon Nov 18 13:29:52 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   63C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
# Install ultralytics
!pip -q install  ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.0/887.0 kB 13.2 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')


!ls /content/drive/MyDrive/Colab\ Notebooks/


Mounted at /content/drive
 b.ipynb
'Copy of Welcome To Colab'
 image_enhancement_working.ipynb
'images (1).zip'
 images.zip
'night_tiling__first_i_have_tried_here_with_11s_now_Rail_Challenge_Starter_(4)_worker3(1040p)(model_saving)_(1)_(1) (2).ipynb'
'Rail_Challenge_Starter_(4)_worker3(1040p)_BETTER_PARAMETERS_(1) (1)UPDATED.ipynb'
 SampleSubmission.csv
'Starter_Notebook (1).ipynb'
 Starter_Notebook.ipynb
 telangana
 Test.csv
 Train.csv


In [5]:
# Import libraries
import pandas as pd
import os
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm
import cv2
import yaml
import matplotlib.pyplot as plt
from ultralytics import YOLO
import multiprocessing

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
# Path to where your data is stored
DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks')

# Preview data files available
os.listdir(DATA_DIR)

['Train.csv',
 'SampleSubmission.csv',
 'Test.csv',
 'Copy of Welcome To Colab',
 'images (1).zip',
 'images.zip',
 'Rail_Challenge_Starter_(4)_worker3(1040p)_BETTER_PARAMETERS_(1) (1)UPDATED.ipynb',
 'night_tiling__first_i_have_tried_here_with_11s_now_Rail_Challenge_Starter_(4)_worker3(1040p)(model_saving)_(1)_(1) (2).ipynb',
 'telangana',
 'Starter_Notebook (1).ipynb',
 'Starter_Notebook.ipynb',
 'b.ipynb',
 'image_enhancement_working.ipynb']

In [7]:
# Set up directoris for training a yolo model

# Images directories
DATASET_DIR = Path('datasets/dataset')
IMAGES_DIR = DATASET_DIR / 'images'
TRAIN_IMAGES_DIR = IMAGES_DIR / 'train'
VAL_IMAGES_DIR = IMAGES_DIR / 'val'
TEST_IMAGES_DIR = IMAGES_DIR / 'test'

# Labels directories
LABELS_DIR = DATASET_DIR / 'labels'
TRAIN_LABELS_DIR = LABELS_DIR / 'train'
VAL_LABELS_DIR = LABELS_DIR / 'val'
TEST_LABELS_DIR = LABELS_DIR / 'test'

In [8]:
# Unzip images to 'images' dir
shutil.unpack_archive(DATA_DIR / 'images.zip', 'images')

In [9]:
import os
import cv2
import torch
import torchvision.transforms as T
from pathlib import Path

# Directories
input_dir = "images"
output_dir = "processed_images"
os.makedirs(output_dir, exist_ok=True)

# Device setup for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Function to enhance image
def enhance_image(image_path):
    try:
        # Load image using OpenCV
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Convert to tensor and move to GPU
        transform = T.Compose([
            T.ToTensor(),
            T.Resize((2200, 2200)),  # Resize if needed
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        img_tensor = transform(img).unsqueeze(0).to(device)

        # Enhance image (dummy example for GPU use)
        enhanced_img = img_tensor * 1.1  # Increase brightness slightly
        enhanced_img = torch.clamp(enhanced_img, 0, 1)

        # Convert back to numpy
        enhanced_img = enhanced_img.squeeze().permute(1, 2, 0).cpu().numpy()
        enhanced_img = (enhanced_img * 255).astype("uint8")

        # Save enhanced image
        enhanced_img = cv2.cvtColor(enhanced_img, cv2.COLOR_RGB2BGR)
        output_path = os.path.join(output_dir, os.path.basename(image_path))
        cv2.imwrite(output_path, enhanced_img)
        print(f"Processed: {image_path}")

        # Delete original image to save space
        os.remove(image_path)
        print(f"Deleted original image: {image_path}")

    except Exception as e:
        print(f"Error processing {image_path}: {e}")

# Batch process images
def batch_process():
    image_files = [str(Path(input_dir) / f) for f in os.listdir(input_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"Found {len(image_files)} images. Starting processing...")
    for image_file in image_files:
        enhance_image(image_file)

if __name__ == "__main__":
    batch_process()


Streaming output truncated to the last 5000 lines.
Processed: images/id_e8xljn.jpg
Deleted original image: images/id_e8xljn.jpg
Processed: images/id_vhb6gt.jpg
Deleted original image: images/id_vhb6gt.jpg
Processed: images/id_rsudhv.jpg
Deleted original image: images/id_rsudhv.jpg
Processed: images/id_tz317c.jpg
Deleted original image: images/id_tz317c.jpg
Processed: images/id_tac1tj.jpg
Deleted original image: images/id_tac1tj.jpg
Processed: images/id_uyhlqj.jpg
Deleted original image: images/id_uyhlqj.jpg
Processed: images/id_egfs4y.jpg
Deleted original image: images/id_egfs4y.jpg
Processed: images/id_4vztwv.jpg
Deleted original image: images/id_4vztwv.jpg
Processed: images/id_qqqs37.jpg
Deleted original image: images/id_qqqs37.jpg
Processed: images/id_uoud66.jpg
Deleted original image: images/id_uoud66.jpg
Processed: images/id_cu5zis.jpg
Deleted original image: images/id_cu5zis.jpg
Processed: images/id_2f1k6g.jpg
Deleted original image: images/id_2f1k6g.jpg
Processed: images/id_f4e6

In [10]:
# Load train and test files
train = pd.read_csv(DATA_DIR / 'Train.csv')
test = pd.read_csv(DATA_DIR / 'Test.csv')
ss = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')

# Add an image_path column
train['image_path'] = [Path('processed_images/' + x) for x in train.Image_ID]
test['image_path'] = [Path('processed_images/' + x) for x in test.Image_ID]

# Map str classes to ints (label encoding targets)
class_mapper = {x:y for x,y in zip(sorted(train['class'].unique().tolist()), range(train['class'].nunique()))}
train['class_id'] = train['class'].map(class_mapper)

# Preview the head of the train set
train.head()

,Image_ID,confidence,class,ymin,xmin,ymax,xmax,image_path,class_id
0,id_11543h.jpg,1.0,Pepper_Bacterial_Spot,194.649671,328.803454,208.107730,341.967928,processed_images/id_11543h.jpg,5
1,id_11543h.jpg,1.0,Pepper_Bacterial_Spot,149.632401,256.768914,162.910362,266.195724,processed_images/id_11543h.jpg,5
2,id_11543h.jpg,1.0,Pepper_Bacterial_Spot,234.046875,327.138158,252.712993,338.876645,processed_images/id_11543h.jpg,5
3,id_11543h.jpg,1.0,Pepper_Bacterial_Spot,221.277138,340.411184,238.593750,354.651316,processed_images/id_11543h.jpg,5
4,id_11ee1c.jpg,1.0,Pepper_Fusarium,2000.563598,989.588908,2184.252196,1401.748952,processed_images/id_11ee1c.jpg,8


In [11]:
test.head()

,Image_ID,confidence,class,ymin,xmin,ymax,xmax,image_path
0,id_128pxx.jpg,NaN,NaN,NaN,NaN,NaN,NaN,processed_images/id_128pxx.jpg
1,id_12jbci.jpg,NaN,NaN,NaN,NaN,NaN,NaN,processed_images/id_12jbci.jpg
2,id_143s4o.jpg,NaN,NaN,NaN,NaN,NaN,NaN,processed_images/id_143s4o.jpg
3,id_14tfmb.jpg,NaN,NaN,NaN,NaN,NaN,NaN,processed_images/id_14tfmb.jpg
4,id_14tw4o.jpg,NaN,NaN,NaN,NaN,NaN,NaN,processed_images/id_14tw4o.jpg


In [12]:
ss.head()

,Image_ID,class,confidence,ymin,xmin,ymax,xmax
0,id_128pxx.jpg,Corn_Cercospora_Leaf_Spot,0.5,100,100,100,100
1,id_128pxx.jpg,Corn_Common_Rust,0.5,100,100,100,100
2,id_128pxx.jpg,Corn_Healthy,0.5,100,100,100,100
3,id_128pxx.jpg,Corn_Northern_Leaf_Blight,0.5,100,100,100,100
4,id_128pxx.jpg,Corn_Streak,0.5,100,100,100,100


In [13]:
# Split data into training and validation
train_unique_imgs_df = train.drop_duplicates(subset = ['Image_ID'], ignore_index = True)
X_train, X_val = train_test_split(train_unique_imgs_df, test_size = 0.25, stratify=train_unique_imgs_df['class'], random_state=42)

X_train = train[train.Image_ID.isin(X_train.Image_ID)]
X_val = train[train.Image_ID.isin(X_val.Image_ID)]

# Check shapes of training and validation data
X_train.shape, X_val.shape

((30777, 9), (10252, 9))

In [14]:
# Preview target distribution, seems there a class imbalance that needs to be handled
X_train['class'].value_counts(normalize = True), X_val['class'].value_counts(normalize = True)

(class
 Corn_Cercospora_Leaf_Spot    0.160444
 Tomato_Septoria              0.159047
 Tomato_Late_Blight           0.098905
 Corn_Streak                  0.077201
 Tomato_Healthy               0.069045
 Pepper_Septoria              0.051922
 Pepper_Leaf_Mosaic           0.051662
 Tomato_Early_Blight          0.047763
 Pepper_Bacterial_Spot        0.047665
 Corn_Common_Rust             0.040290
 Pepper_Leaf_Curl             0.037561
 Corn_Healthy                 0.037561
 Tomato_Fusarium              0.019950
 Pepper_Healthy               0.017935
 Pepper_Late_Blight           0.014296
 Pepper_Leaf_Blight           0.012574
 Tomato_Bacterial_Spot        0.011860
 Pepper_Cercospora            0.011405
 Pepper_Fusarium              0.011340
 Tomato_Leaf_Curl             0.011177
 Corn_Northern_Leaf_Blight    0.004289
 Tomato_Mosaic                0.003314
 Pepper_Early_Blight          0.002794
 Name: proportion, dtype: float64,
 class
 Corn_Cercospora_Leaf_Spot    0.156067
 Tomato_Septori

In [15]:
# Check if dirs exist, if they do, remove them, otherwise create them.
# This only needs to run once
for DIR in [TRAIN_IMAGES_DIR,VAL_IMAGES_DIR, TEST_IMAGES_DIR, TRAIN_LABELS_DIR,VAL_LABELS_DIR,TEST_LABELS_DIR]:
  if DIR.exists():
    shutil.rmtree(DIR)
  DIR.mkdir(parents=True, exist_ok = True)

In [16]:
# Copy train, val and test images to their respective dirs
for img in tqdm(X_train.image_path.unique()):
  shutil.copy(img, TRAIN_IMAGES_DIR / img.parts[-1])

for img in tqdm(X_val.image_path.unique()):
  shutil.copy(img, VAL_IMAGES_DIR / img.parts[-1])

for img in tqdm(test.image_path.unique()):
  shutil.copy(img, TEST_IMAGES_DIR / img.parts[-1])

  0%|          | 0/3676 [00:00<?, ?it/s]

  0%|          | 0/1226 [00:00<?, ?it/s]

  0%|          | 0/2101 [00:00<?, ?it/s]

In [26]:
X_train.head()

,Image_ID,confidence,class,ymin,xmin,ymax,xmax,image_path,class_id
0,id_11543h.jpg,1.0,Pepper_Bacterial_Spot,194.649671,328.803454,208.107730,341.967928,processed_images/id_11543h.jpg,5
1,id_11543h.jpg,1.0,Pepper_Bacterial_Spot,149.632401,256.768914,162.910362,266.195724,processed_images/id_11543h.jpg,5
2,id_11543h.jpg,1.0,Pepper_Bacterial_Spot,234.046875,327.138158,252.712993,338.876645,processed_images/id_11543h.jpg,5
3,id_11543h.jpg,1.0,Pepper_Bacterial_Spot,221.277138,340.411184,238.593750,354.651316,processed_images/id_11543h.jpg,5
6,id_11gglx.jpg,1.0,Corn_Cercospora_Leaf_Spot,774.562500,2735.933839,850.476742,2834.348725,processed_images/id_11gglx.jpg,0


In [ ]:
import multiprocessing
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm
import shutil
import pandas as pd

# Function to convert the bboxes to YOLO format
def convert_to_yolo(bbox, width, height):
    ymin, xmin, ymax, xmax = bbox['ymin'], bbox['xmin'], bbox['ymax'], bbox['xmax']
    class_id = bbox['class_id']

    # Normalize the coordinates
    x_center = (xmin + xmax) / 2 / width
    y_center = (ymin + ymax) / 2 / height
    bbox_width = (xmax - xmin) / width
    bbox_height = (ymax - ymin) / height

    return f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}"

# Top-level function to save annotations for a single image
def save_yolo_annotations_task(task):
    image_path, bboxes, output_dir = task
    try:
        img = np.array(Image.open(str(image_path)))
        height, width, _ = img.shape
    except Exception as e:
        print(f"Error opening image {image_path}: {e}")
        return

    label_file = Path(output_dir) / f"{Path(image_path).stem}.txt"
    with open(label_file, 'w') as f:
        for bbox in bboxes:
            annotation = convert_to_yolo(bbox, width, height)
            f.write(f"{annotation}\n")

# Function to clear output directory
def clear_output_dir(output_dir):
    if Path(output_dir).exists():
        shutil.rmtree(output_dir)
    Path(output_dir).mkdir(parents=True, exist_ok=True)

# Function to process the dataset and save annotations
def process_dataset(dataframe, output_dir):
    # Clear the output directory to prevent duplicate annotations
    clear_output_dir(output_dir)

    # Group the DataFrame by 'image_path'
    grouped = dataframe.groupby('image_path')
    tasks = [(image_path, group.to_dict('records'), output_dir) for image_path, group in grouped]

    # Use multiprocessing Pool to process tasks
    with multiprocessing.Pool() as pool:
        list(tqdm(pool.imap_unordered(save_yolo_annotations_task, tasks), total=len(tasks)))


# Save train and validation labels to their respective dirs
process_dataset(X_train, TRAIN_LABELS_DIR)
process_dataset(X_val, VAL_LABELS_DIR)

In [ ]:
# Train images dir
TRAIN_IMAGES_DIR

In [ ]:
# Create a data.yaml file required by yolo
class_names = sorted(train['class'].unique().tolist())
num_classes = len(class_names)

data_yaml = {
    'train': '/content/' + str(TRAIN_IMAGES_DIR),
    'val': '/content/' + str(VAL_IMAGES_DIR),
    'test': '/content/' + str(TEST_IMAGES_DIR),
    'nc': num_classes,
    'names': class_names
}

yaml_path = 'data.yaml'
with open(yaml_path, 'w') as file:
    yaml.dump(data_yaml, file, default_flow_style=False)

# Preview data yaml file
data_yaml

In [ ]:
# Plot some images and their bboxes to ensure the conversion was done correctly
def load_annotations(label_path):
    with open(label_path, 'r') as f:
        lines = f.readlines()
    boxes = []
    for line in lines:
        class_id, x_center, y_center, width, height = map(float, line.strip().split())
        boxes.append((class_id, x_center, y_center, width, height))
    return boxes

# Function to plot an image with its bounding boxes
def plot_image_with_boxes(image_path, boxes):
    # Load the image
    image = np.array(Image.open(str(image_path)))


    # Get image dimensions
    h, w, _ = image.shape

    # Plot the image
    plt.figure(figsize=(10, 10))
    plt.imshow(image)

    # Plot each bounding box
    for box in boxes:
        class_id, x_center, y_center, width, height = box
        # Convert YOLO format to corner coordinates
        xmin = int((x_center - width / 2) * w)
        ymin = int((y_center - height / 2) * h)
        xmax = int((x_center + width / 2) * w)
        ymax = int((y_center + height / 2) * h)

        # Draw the bounding box
        plt.gca().add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                          edgecolor='red', facecolor='none', linewidth=2))
        plt.text(xmin, ymin - 10, f'Class {int(class_id)}', color='red', fontsize=8, weight='bold')

    plt.axis('off')
    plt.show()

# Directories for images and labels
IMAGE_DIR = TRAIN_IMAGES_DIR
LABEL_DIR = TRAIN_LABELS_DIR

# Plot a few images with their annotations
for image_name in os.listdir(IMAGE_DIR)[:5]:
    image_path = IMAGE_DIR / image_name
    label_path = LABEL_DIR / (image_name.replace('.jpg', '.txt').replace('.png', '.txt'))

    if label_path.exists():
        boxes = load_annotations(label_path)
        print(f"Plotting {image_name} with {len(boxes)} bounding boxes.")
        plot_image_with_boxes(image_path, boxes)
    else:
        print(f"No annotations found for {image_name}.")


In [ ]:
from ultralytics import YOLO  # Assuming the YOLO framework is installed


# Load a yolo pretrained model
model = YOLO('yolo11s.pt')

# Fine tune model to our data
model.train(
    data='data.yaml',          # Path to the dataset configuration
    epochs=20,                 # Number of epochs
    imgsz=2000,                # Image size (height, width)
    batch=5,                   # Batch size
    device=0,                  # Device to use (0 for the first GPU)
    amp=True,                 # Enable mixed precision training
    patience=5,
    workers=28)

In [ ]:
# Validate the model on the validation set
model = YOLO('/content/runs/detect/train/weights/best.pt')
results = model.val()

In [ ]:
# Assuming 'model' is your trained YOLO model
model.savefiles.download('/content/image processing.pt')
  # Save model as .pt file

from google.colab import files
files.download('/content/image processing.pt')


In [ ]:
# Load the trained YOLO model

# Load test data from test.csv
test = pd.read_csv(DATA_DIR / 'Test.csv')
test_dir_path = '/content/datasets/dataset/images/test'

# Initialize an empty list to store results
all_data = []
max_attempts = 1  # Define the maximum number of attempts per image
no_detection_count = 0  # Counter for images with no detections after all attempts

# Iterate through each image listed in test.csv
for image_id in tqdm(test.Image_ID):
    img_path = os.path.join(test_dir_path, image_id)

    # Check if the image exists before running prediction
    if os.path.exists(img_path):
        attempt = 0
        detected = False

        # Retry loop
        while attempt < max_attempts and not detected:
            attempt += 1
            results = model(img_path)

            # Check if there are any detections
            if results[0].boxes:
                boxes = results[0].boxes.xyxy.tolist()
                classes = results[0].boxes.cls.tolist()
                confidences = results[0].boxes.conf.tolist()
                names = results[0].names

                # Append actual detections to the results list
                for box, cls, conf in zip(boxes, classes, confidences):
                    x1, y1, x2, y2 = box
                    detected_class = names[int(cls)]
                    all_data.append({
                        'Image_ID': image_id,
                        'class': detected_class,
                        'confidence': conf,
                        'ymin': y1,
                        'xmin': x1,
                        'ymax': y2,
                        'xmax': x2
                    })
                detected = True  # Stop further attempts for this image once detected
            else:
                print(f"Attempt {attempt} for {image_id} yielded no detections.")

        # If still no detection after max attempts
        if not detected:
            no_detection_count += 1  # Increment the no-detection counter
            all_data.append({
                'Image_ID': image_id,
                'class': '',  # Or keep blank if preferred
                'confidence': 0.0,
                'ymin': None,
                'xmin': None,
                'ymax': None,
                'xmax': None
            })
    else:
        print(f"Warning: Image {image_id} does not exist in the directory {test_dir_path}")

# Convert to DataFrame
sub = pd.DataFrame(all_data)

# Save to submission.csv
sub.to_csv('i_don_try_thankGod_pre_image_processinf.csv', index=False)

# Print the total number of images with no detections
print(f"Total number of images with no detections after {max_attempts} attempts: {no_detection_count}")


In [ ]:

from google.colab import files


files.download('i_don_try_thankGod_pre_image_processinf.csv')

In [ ]:
# Assuming 'model' is your trained YOLO model
model.save('/content/image processing.pt')
  # Save model as .pt file

from google.colab import files
files.download('/content/image processing.pt')
